In [1]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolPROD.ini')

dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [2]:
QUERY = connect.execute("""SELECT * FROM 

(
            

SELECT DISTINCT LH.HOTEL_ID, 
                LHC.HOTEL_CITY_NAME_EN ,
                LHCo.HOTEL_COUNTRY_NAME ,
            LH.HOTEL_STATUS_ID,
            LHS.HOTEL_STATUS_SHORT_NAME,
            SPEND_DATA.CNT_ARRIVAL_DATE AS SD_CNT_ARRIVAL_DATE,
            HSV.HSV_CNT_TRAVEL_DATE AS HSV_CNT_TRAVEL_DATE,
            BKG.BKG_CNT_ARRIVAL_DATE AS BKG_CNT_ARRIVAL_DATE,
            CRO_DATA_NON_CONTRACTED.CRO_NON_CONTRACTED_CNT_TRAVEL_DATE AS CRO_NON_CONTRACTED_CNT_TRAVEL_DATE,
             COALESCE(CRO_DATA_CONTRACTED.IS_CCR_HOTEL,0) AS IS_CCR_HOTEL ,
             CASE WHEN ERFP_MP.HOTEL_ID IS NOT NULL THEN 1 ELSE 0 END AS IS_MP_HOTEL,
             CASE WHEN FRG.HOTEL_ID IS NOT NULL THEN 1 ELSE 0 END AS IS_RG_HOTEL,
             ((COALESCE(LOCAL.SD_CNT_ARRIVAL_DATE,0)+
            COALESCE(LOCAL.HSV_CNT_TRAVEL_DATE,0)+
            COALESCE(LOCAL.BKG_CNT_ARRIVAL_DATE,0)+
            COALESCE(LOCAL.CRO_NON_CONTRACTED_CNT_TRAVEL_DATE,0)))*100/(5*365)
            
            
            
            
            
FROM DWHBIL.V_LKP_HOTEL LH
LEFT JOIN DWHBIL.LKP_ERFP_MARKET_PLACE ERFP_MP ON ERFP_MP.HOTEL_ID = LH.HOTEL_ID AND YEAR(ERFP_MP.SIGN_UP_DATE) BETWEEN year(CURRENT_DATE)-1 AND year(CURRENT_DATE)
LEFT JOIN DWHBIL.FAK_RATEGAIN FRG ON FRG.HOTEL_ID = LH.HOTEL_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_RATING LHRT ON LHRT.HOTEL_ID = LH.HOTEL_ID AND  LHRT.RATING_CREATION_DATE BETWEEN CURDATE()-730 AND CURDATE() AND IS_ACTIVE_RATING = 1
LEFT JOIN DWHBIL.LKP_HOTEL_GOOGLE_DETAIL LHGD ON LHGD.HOTEL_ID = LH.HOTEL_ID
JOIN DWHBIL.LKP_HOTEL_CATEGORY LHCT ON LHCT.HOTEL_CATEGORY_ID = LH.HOTEL_CATEGORY_ID
JOIN DWHBIL.V_LKP_HOTEL_CITY LHC ON LH.HOTEL_CITY_ID = LHC.HOTEL_CITY_ID
JOIN DWHBIL.V_LKP_HOTEL_COUNTRY LHCo ON LHCo.HOTEL_COUNTRY_ID = LH.HOTEL_COUNTRY_ID
JOIN DWHBIL.LKP_HOTEL_STATUS LHS ON LH.HOTEL_STATUS_ID = LHS.HOTEL_STATUS_ID
LEFT JOIN 

--Spend Data

(
SELECT  FR_RATE.HOTEL_ID AS HOTEL_ID
        ,COUNT(distinct FR_RATE.ARRIVAL_DATE )AS CNT_ARRIVAL_DATE
FROM DWHBIL.V_LKP_HOTEL HOTEL
LEFT JOIN 
 
      (         
                SELECT  SR.HOTEL_ID 
                 ,F.FILE_NAME AS FILE_NAME
                 ,SR.NUMBER_ROOMNIGHTS AS NUMBER_ROOM_NIGHTS
                 ,SR.AMOUNT_TURNOVER_TOTAL_GROSS_REPORT_CURRENCY AS TURNOVER
                 ,H.HOTEL_CITY_ID
                 ,H.HOTEL_COUNTRY_ID
                 ,H.HOTEL_CATEGORY_ID
                 ,SR.ARRIVAL_DATE
                 ,ROUND((LOCAL.TURNOVER/LOCAL.NUMBER_ROOM_NIGHTS),2) AS TADR
                FROM DWHBIL.FAK_EXTERNAL_SPEND_RAW SR
                JOIN DWHBIL.V_LKP_HOTEL H ON H.HOTEL_ID = SR.HOTEL_ID
                JOIN DWHBIL.FAK_SDF_FILE_QS F ON F.FILE_ID = SR.FILE_ID 
                WHERE SR.ARRIVAL_DATE BETWEEN CURRENT_DATE - 365 AND CURRENT_DATE
                AND LOCAL.NUMBER_ROOM_NIGHTS > 0 AND LOCAL.TURNOVER > 0 AND SR.HOTEL_ID > 0

       ) FR_RATE ON FR_RATE.HOTEL_ID = HOTEL.HOTEL_ID
LEFT JOIN DWHBIL.REL_FOCUS_DESTINATION_ANALYSIS_TO_FILENAME DEST_ANAL_NEW ON FR_RATE.FILE_NAME = DEST_ANAL_NEW.FILE_NAME
LEFT JOIN DWHBIL.LKP_DESTINATION_ANALYSIS_NAME DEST_ANAL_NAME ON DEST_ANAL_NEW.DESTINATION_ANALYSIS_NAME = DEST_ANAL_NAME.DESTINATION_ANALYSIS_NAME AND DEST_ANAL_NAME.IS_VALID_DESTINATION_ANALYSIS =1  
GROUP BY LOCAL.HOTEL_ID
         
) SPEND_DATA ON SPEND_DATA.HOTEL_ID = LH.HOTEL_ID

--HSV

LEFT JOIN 
(

          SELECT  HSV.HOTEL_ID,
            COUNT(distinct  HSV.TRAVEL_DATE) AS HSV_CNT_TRAVEL_DATE
          FROM DWHBIL.FAK_HSV3_STD_RATE_BY_TRAVEL_DATE HSV 
          JOIN DWHBIL.LKP_BOOKING_RATE_TYPE RT ON HSV.RATE_TYPE_ID = RT.BOOKING_RATE_TYPE_ID 
          WHERE  HSV.TRAVEL_DATE  BETWEEN CURDATE() - 365 AND CURDATE() 
           AND BOOKING_RATE_GROUP_ID NOT IN (10,14,6,7,2) AND  ROOM_TYPE_ID IN (0,1) 
          GROUP BY 
          HSV.HOTEL_ID
) HSV ON HSV.HOTEL_ID = LH.HOTEL_ID

--BKG

LEFT JOIN 
(

         SELECT  HO.HOTEL_ID,
           COUNT(DISTINCT BKG.HOTEL_ARRIVAL_DATE) BKG_CNT_ARRIVAL_DATE
         FROM DWHBIL.FAK_BOOKING BKG
         JOIN DWHBIL.V_LKP_HOTEL HO ON BKG.HOTEL_ID = HO.HOTEL_ID
         WHERE (BKG.HOTEL_ARRIVAL_DATE) BETWEEN CURDATE() - 365 AND CURDATE()
         AND BKG.BOOKING_STATUS_ID IN (0,1)
         AND BKG.BOOKING_SOURCE_ID <> 742
         AND NUMBER_BOOKED_ROOMNIGHTS > 0
         AND HO.HOTEL_COUNTRY_ID > 0
         AND HO.HOTEL_CATEGORY_ID>0
         AND HO.HOTEL_CITY_ID >0
         AND HO.HOTEL_ID > 0
         GROUP BY   HO.HOTEL_ID


) BKG ON BKG.HOTEL_ID = LH.HOTEL_ID

--CRO_CONTRACTED

LEFT JOIN 
(

         SELECT  CRO.HOTEL_ID, 
         1 AS IS_CCR_HOTEL
           FROM DWHBIL.FAK_CRO_RATE CRO
         WHERE   YEAR(CRO.TRAVEL_DATE) BETWEEN YEAR(CURRENT_DATE)-1 AND YEAR(CURRENT_DATE)
         AND CRO.HOTEL_ROOM_CATEGORY_ID = 0 AND 
         CRO.ROOM_TYPE_ID IN (0,100)  AND CRO.F_KEY = 12538 
         GROUP BY  (CRO.HOTEL_ID)

) CRO_DATA_CONTRACTED ON CRO_DATA_CONTRACTED.HOTEL_ID = LH.HOTEL_ID

--CRO_NON_CONTRACTED
LEFT JOIN 
(

         SELECT  CRO.HOTEL_ID, 
           COUNT(DISTINCT CRO.TRAVEL_DATE) CRO_NON_CONTRACTED_CNT_TRAVEL_DATE
          FROM DWHBIL.FAK_CRO_RATE CRO
         WHERE   YEAR(CRO.TRAVEL_DATE) BETWEEN YEAR(CURRENT_DATE)-1 AND YEAR(CURRENT_DATE)
         AND CRO.HOTEL_ROOM_CATEGORY_ID = 0 AND 
         CRO.ROOM_TYPE_ID IN (0,100)  AND CRO.F_KEY <> 12538 
         GROUP BY  (CRO.HOTEL_ID)

) CRO_DATA_NON_CONTRACTED ON CRO_DATA_NON_CONTRACTED.HOTEL_ID = LH.HOTEL_ID
) A
""")
import pandas as pd
df = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
df = pd.DataFrame(a)
df_col_names = QUERY.col_names
df.columns = df_col_names
df.count()
# import datetime
 
# d = datetime.datetime.today()
# df.to_excel('C:\\Users\\svi02\\Documents\\raw_data'+str(d.strftime('%d-%m-%Y_%H%M'))+'.xlsx', 
#           sheet_name='Results', 
#           header = True,
#           encoding='utf-8',
#           index=False)


HOTEL_ID                                                                                                                            756136
HOTEL_CITY_NAME_EN                                                                                                                  418718
HOTEL_COUNTRY_NAME                                                                                                                  756136
HOTEL_STATUS_ID                                                                                                                     756136
HOTEL_STATUS_SHORT_NAME                                                                                                             756136
SD_CNT_ARRIVAL_DATE                                                                                                                  11638
HSV_CNT_TRAVEL_DATE                                                                                                                  72127
BKG_CNT_ARRIVAL_DATE       

In [3]:
df.to_csv('C:\\Users\\svi02\\Documents\\Rate_gain.csv', 
          sep = ',', 
          header = True,
          encoding='utf-8',
          index=False)

In [4]:
df.to_excel(r'C:\Users\svi02\Documents\misc\Rate_gain.xlsx')

15